# Experiment 05 — Dataset Size Scaling (QNG Optimized)

**Research question:** How does the amount of training data affect VQC performance when using the Quantum Natural Gradient (QNG)?

**Setup:** Fixed 2 features (cols 26, 4), 2 layers, angle encoding.  
We vary the total dataset size: **500 → 1000 → 5000 → 10000**.

### Why this matters
- The paper notes that VQCs may be particularly advantageous with *limited data*
- Quantum models have fewer parameters → potentially better generalisation at small N
- At large N, classical models usually dominate due to scalability

| Dataset Size | Train | Val | Test |
|-------------|-------|-----|------|
| 500 | 300 | 100 | 100 |
| 1,000 | 600 | 200 | 200 |
| 5,000 | 3,000 | 1,000 | 1,000 |
| 10,000 | 6,000 | 2,000 | 2,000 |

In [1]:
import sys
sys.path.append('..')

import numpy as np
import time
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score, roc_curve

from utils.data_utils import load_higgs, binary_accuracy

np.random.seed(42)

## 1. Setup

In [2]:
# Fixed architecture
N_FEATURES = 2
N_LAYERS   = 2
N_EPOCHS   = 30
BATCH_SIZE = 32
LR         = 0.05

DATASET_SIZES = [500, 1000, 5000, 10000]

## 2. Training Engine (QNG)

In [3]:
def train_vqc_qng(X_train, y_train, X_val, y_val, n_epochs, batch_size, lr):
    dev = qml.device('default.qubit', wires=N_FEATURES)

    @qml.qnode(dev, interface='autograd')
    def circuit(weights, x):
        for i in range(N_FEATURES):
            qml.RY(x[i], wires=i)
        for l in range(N_LAYERS):
            for q in range(N_FEATURES):
                qml.Rot(weights[l, q, 0], weights[l, q, 1], weights[l, q, 2], wires=q)
            qml.CNOT(wires=[0, 1])
            qml.CNOT(wires=[1, 0])
        return qml.expval(qml.PauliZ(0))

    pnp.random.seed(42)
    w = pnp.array(np.random.uniform(0, 2*np.pi, (N_LAYERS, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)

    opt_w = qml.QNGOptimizer(stepsize=lr)
    opt_b = qml.AdamOptimizer(stepsize=lr)
    mt_fn = qml.metric_tensor(circuit, approx='block-diag')

    val_losses = []

    for epoch in range(n_epochs):
        perm = np.random.permutation(len(X_train))
        X_shuf, y_shuf = X_train[perm], y_train[perm]

        for start in range(0, len(X_train), batch_size):
            Xb = X_shuf[start:start+batch_size]
            yb = y_shuf[start:start+batch_size].astype(float)

            def cost_w(weights):
                preds = pnp.array([circuit(weights, x) for x in Xb])
                return pnp.mean((yb - (preds + b)) ** 2)
            
            mt = mt_fn(w, Xb[0])
            w = opt_w.step(cost_w, w, metric_tensor_fn=lambda w: mt)

            def cost_b(bias):
                preds = pnp.array([circuit(w, x) for x in Xb])
                return pnp.mean((yb - (preds + bias)) ** 2)
            b = opt_b.step(cost_b, b)

        # Validation metrics
        val_preds = np.array([float(circuit(w, x) + b) for x in X_val])
        vl_loss = np.mean((y_val.astype(float) - val_preds)**2)
        val_losses.append(vl_loss)

    return circuit, w, b, val_losses

## 3. Run Experiments

In [4]:
results = {}

for n_samples in DATASET_SIZES:
    print(f'\n{"="*50}')
    print(f'Running: {n_samples} samples')

    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path='../data/HIGGS.csv.gz',
        n_samples=n_samples,
        n_features=N_FEATURES,
        feature_indices=[26, 4],
        scale_range=(0, np.pi),
)

    t0 = time.time()
    circuit_fn, w_fin, b_fin, v_hist = train_vqc_qng(X_tr, y_tr, X_vl, y_vl, N_EPOCHS, BATCH_SIZE, LR)
    wall_time = time.time() - t0

    # Test evaluation
    test_raw = np.array([float(circuit_fn(w_fin, x) + b_fin) for x in X_te])
    test_acc = binary_accuracy(y_te, test_raw)
    y_test_01 = (y_te == 1).astype(int)
    test_score = (test_raw - test_raw.min()) / (test_raw.max() - test_raw.min() + 1e-8)
    test_auc = roc_auc_score(y_test_01, test_score)

    results[n_samples] = {
        'val_losses': v_hist,
        'test_acc': test_acc,
        'test_auc': test_auc,
        'test_scores': test_score,
        'y_test_01': y_test_01,
        'wall_time': wall_time
    }
    print(f'  → Test acc: {test_acc:.4f} | Test AUC: {test_auc:.4f} | Time: {wall_time:.1f}s')


Running: 500 samples
Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 500 samples | 2 features | train=300, val=100, test=100
  → Test acc: 0.6800 | Test AUC: 0.6700 | Time: 41.9s

Running: 1000 samples
Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 1000 samples | 2 features | train=600, val=200, test=200
  → Test acc: 0.6800 | Test AUC: 0.6664 | Time: 83.1s

Running: 5000 samples
Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 5000 samples | 2 features | train=3000, val=1000, test=1000
  → Test acc: 0.5990 | Test AUC: 0.6245 | Time: 411.7s

Running: 10000 samples
Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 10000 samples | 2 features | train=6000, val=2000, test=2000
  → Test acc: 0.5970 | Test AUC: 0.5914 | Time: 827.1s


## 4. Results Summary

In [5]:
print('\nDataset Size Scaling Results (QNG Optimized)')
print(f'{"Samples":>10} {"Test Acc":>10} {"Test AUC":>10} {"Time(s)":>10}')
print('-' * 45)
for ns, r in results.items():
    print(f'{ns:>10} {r["test_acc"]:>10.4f} {r["test_auc"]:>10.4f} {r["wall_time"]:>10.1f}')


Dataset Size Scaling Results (QNG Optimized)
   Samples   Test Acc   Test AUC    Time(s)
---------------------------------------------
       500     0.6800     0.6700       41.9
      1000     0.6800     0.6664       83.1
      5000     0.5990     0.6245      411.7
     10000     0.5970     0.5914      827.1
